# Measuring Performance

**IMSE 774 · Neural Networks · Week 8 · Sessions 15–16 · October 13 & 15, 2026**

Every code cell from the [lecture note](https://harunpirim.github.io/IMSE774-NeuralNetworks/notes/08-measuring-performance.html), in the order it appears
there. The prose, figures, and exercises live on that page; this notebook is
for running and changing the code.

Run a cell with **Shift+Enter**. Run the setup cell first — the later cells
depend on the names it defines.

Source: [https://github.com/harunpirim/IMSE774-NeuralNetworks](https://github.com/harunpirim/IMSE774-NeuralNetworks)


## Setup


`setup`


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

torch.manual_seed(774)
np.random.seed(774)

plt.rcParams.update({
    "figure.figsize": (7, 4),
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.frameon": False,
})

NDSU_GREEN = "#005643"
NDSU_GOLD  = "#FFC72C"
ACCENT     = "#C1121F"
STEEL      = "#1864AB"

## Training a Simple Model


`fig-overconfident`


In [ ]:
g = torch.Generator().manual_seed(774)
K, D_i = 10, 40
class_means = 0.33 * torch.randn(K, D_i, generator=g)

def make_data(n):
    """n examples: a random class label, then that class's mean plus unit Gaussian noise."""
    y = torch.randint(0, K, (n,), generator=g)
    return class_means[y] + torch.randn(n, D_i, generator=g), y

x_train, y_train = make_data(400)
x_test,  y_test  = make_data(2000)

# With equal priors and identical isotropic noise, the nearest class mean is the
# Bayes-optimal classifier, so its error is the noise floor for this problem.
dist = ((x_test[:, None, :] - class_means[None]) ** 2).sum(-1)
bayes_error = 100 * (dist.argmin(1) != y_test).float().mean().item()

torch.manual_seed(774)
model = nn.Sequential(nn.Linear(D_i, 100), nn.ReLU(),
                      nn.Linear(100, 100), nn.ReLU(),
                      nn.Linear(100, K))
opt = torch.optim.SGD(model.parameters(), lr=0.05)
criterion = nn.CrossEntropyLoss()

log = []                                                # step, train error, train loss, test error, test loss
for step in range(3001):
    if step % 20 == 0:
        with torch.no_grad():
            row = [step]
            for x, y in [(x_train, y_train), (x_test, y_test)]:
                out = model(x)
                row += [100 * (out.argmax(1) != y).float().mean().item(), criterion(out, y).item()]
            log.append(row)
    idx = torch.randint(0, 400, (100,), generator=g)
    loss = criterion(model(x_train[idx]), y_train[idx])
    opt.zero_grad(); loss.backward(); opt.step()
log = np.array(log)
best = log[log[:, 4].argmin()]

fig, ax = plt.subplots(1, 2, figsize=(8, 3.2))
ax[0].plot(log[:, 0], log[:, 1], color=ACCENT, label="Train")
ax[0].plot(log[:, 0], log[:, 3], color=NDSU_GREEN, label="Test")
ax[0].axhline(bayes_error, ls="--", color="gray", lw=1, label="Best possible")
ax[0].set(xlabel="Training step", ylabel="% error", ylim=(0, 100))
ax[0].legend()
ax[1].plot(log[:, 0], log[:, 2], color=ACCENT, label="Train")
ax[1].plot(log[:, 0], log[:, 4], color=NDSU_GREEN, label="Test")
ax[1].axvline(best[0], ls=":", color="gray", lw=1)
ax[1].set(xlabel="Training step", ylabel="Cross-entropy loss")
fig.tight_layout()
plt.show()

print(f"best achievable test error (nearest class mean): {bayes_error:.1f}%")
print(f"lowest test loss {best[4]:.3f} at step {best[0]:.0f}, where test error is {best[3]:.1f}%")
print(f"after 3,000 steps: train error {log[-1, 1]:.1f}%, train loss {log[-1, 2]:.4f}; "
      f"test error {log[-1, 3]:.1f}%, test loss {log[-1, 4]:.3f}")

## Mathematical formulation of test error


`decomposition-check`


In [ ]:
SIGMA = 0.15                                            # noise standard deviation

def mu(x):
    """Our quasi-sinusoidal ground truth (Prince's figure 8.3 uses a similar curve)."""
    return 0.5 * (np.exp(np.sin(2 * np.pi * x)) - 1.27)

def sample(I, rng):
    """One x_i drawn uniformly from each of I equal segments of [0, 1], as in figure 8.3."""
    x = (np.arange(I) + rng.uniform(size=I)) / I
    return x, mu(x) + SIGMA * rng.standard_normal(I)

def features(x, D):
    """Figure 8.4: a column of ones (for beta) plus D slope-one ReLUs with joints at 0, 1/D, ..., (D-1)/D."""
    joints = np.arange(D) / D
    return np.hstack([np.ones((len(x), 1)), np.maximum(x[:, None] - joints, 0)])

def fit(x, y, D):
    """Closed-form least squares (problem 8.3). When D + 1 > I, lstsq returns the minimum-norm solution."""
    return np.linalg.lstsq(features(x, D), y, rcond=None)[0]

def predict(phi, x, D):
    return features(x, D) @ phi

x_grid = np.linspace(0, 1, 501)                         # test inputs spread over [0, 1]

def bias_variance(D, I, R=1000, seed=774):
    """Estimate the bias and variance terms of equation (8.7), averaged over x, from R training sets."""
    rng = np.random.default_rng(seed)
    preds = np.array([predict(fit(*sample(I, rng), D), x_grid, D) for _ in range(R)])
    f_mu = preds.mean(axis=0)                           # equation (8.4)
    return np.mean((f_mu - mu(x_grid)) ** 2), np.mean(preds.var(axis=0))

# Left side of (8.7): train on a fresh dataset, test against fresh noisy outputs, average.
rng = np.random.default_rng(774)
I, D, R = 15, 3, 4000
test_loss = np.empty(R)
for r in range(R):
    phi = fit(*sample(I, rng), D)
    y_new = mu(x_grid) + SIGMA * rng.standard_normal(x_grid.size)
    test_loss[r] = np.mean((predict(phi, x_grid, D) - y_new) ** 2)

b, v = bias_variance(D, I, R)
print(f"expected test loss, by simulation:     {test_loss.mean():.4f}")
print(f"variance {v:.4f} + bias {b:.4f} + noise {SIGMA**2:.4f} = {v + b + SIGMA**2:.4f}")

## The bias–variance trade-off


`fig-bias-variance`


In [ ]:
caps = np.arange(1, 13)
bv = np.array([bias_variance(D, 15) for D in caps])     # columns: bias, variance

fig, ax = plt.subplots(1, 2, figsize=(8, 3.2))
ax[0].plot(caps, bv[:, 0], color=ACCENT, label="bias")
ax[0].plot(caps, bv[:, 1], color=NDSU_GREEN, label="variance")
ax[0].plot(caps, bv.sum(1), "--", color="gray", label="bias + variance")
ax[0].set(xlabel="Capacity $D$ (hidden units)", ylabel="Mean squared error", ylim=(0, 0.1), title="$I$ = 15")
ax[0].legend()

print(f"{'I':>5}{'best D':>8}{'bias + variance':>17}")
for I, c in [(15, NDSU_GREEN), (50, STEEL), (200, ACCENT)]:
    total = np.array([sum(bias_variance(D, I, R=400)) for D in caps])
    ax[1].plot(caps, total, color=c, label=f"$I$ = {I}")
    ax[1].plot(caps[total.argmin()], total.min(), "o", color=c)
    print(f"{I:>5}{caps[total.argmin()]:>8}{total.min():>17.4f}")
ax[1].set(yscale="log", xlabel="Capacity $D$", ylabel="Bias + variance", title="Three training-set sizes")
ax[1].legend()
fig.tight_layout()
plt.show()

## Explanation


`high-dimensions`


In [ ]:
rng = np.random.default_rng(774)
print(f"{'dim':>5}{'mean |x|':>10}{'sd |x|':>8}{'sqrt(dim)':>11}{'mean |cos|':>12}{'nearest/farthest':>18}")
for dim in [2, 10, 40, 100, 1000]:
    X = rng.standard_normal((1000, dim))                # 1,000 standard-normal samples
    r = np.linalg.norm(X, axis=1)                       # distance from the origin
    U = X / r[:, None]
    cos = U[:500] @ U[500:].T                           # angles between 250,000 random pairs
    d = np.linalg.norm(X - rng.standard_normal(dim), axis=1)   # distances to one new query point
    print(f"{dim:>5}{r.mean():>10.2f}{r.std():>8.2f}{np.sqrt(dim):>11.2f}"
          f"{np.abs(cos).mean():>12.3f}{d.min() / d.max():>18.3f}")

`fig-double-descent`


In [ ]:
I = 15
D_list = np.unique(np.r_[np.round(np.logspace(0, 3, 40)).astype(int), np.arange(10, 21)])
rng = np.random.default_rng(774)
datasets = [sample(I, rng) for _ in range(200)]
err = np.array([[np.mean((predict(fit(x, y, D), x_grid, D) - mu(x_grid)) ** 2) for x, y in datasets]
                for D in D_list])                       # rows: capacity; columns: training set
med = np.median(err, axis=1)
q1, q3 = np.percentile(err, [25, 75], axis=1)

fig, ax = plt.subplots(1, 2, figsize=(8, 3.2))
ax[0].fill_between(D_list, q1, q3, color=NDSU_GREEN, alpha=0.2, lw=0)
ax[0].plot(D_list, med, color=NDSU_GREEN)
ax[0].axvline(I - 1, ls="--", color="gray", lw=1)
ax[0].text(I - 1, ax[0].get_ylim()[1], " $D+1=I$", va="top", fontsize=8, color="gray")
ax[0].set(xscale="log", yscale="log", xlabel="Capacity $D$ (hidden units)", ylabel="Error vs. true function")

x, y = datasets[0]
ax[1].plot(x_grid, mu(x_grid), color="k", lw=2, label="true $\\mu[x]$")
for D, c in [(4, STEEL), (I - 1, ACCENT), (1000, NDSU_GREEN)]:
    ax[1].plot(x_grid, predict(fit(x, y, D), x_grid, D), color=c, lw=1.4, label=f"$D$ = {D}")
ax[1].plot(x, y, "o", color=NDSU_GOLD, mec="k", ms=5, zorder=5)
ax[1].set(ylim=(-1.2, 1.2), xlabel="Input, $x$", ylabel="Output, $y$")
ax[1].legend(fontsize=8, ncol=2, loc="lower left")
fig.tight_layout()
plt.show()

classical = D_list < I - 1
print(f"classical minimum: D = {D_list[classical][med[classical].argmin()]}, error {med[classical].min():.4f}")
print(f"peak:              D = {D_list[med.argmax()]}, error {med.max():.4f}")
print(f"modern regime:     D = {D_list[-1]}, error {med[-1]:.4f}")

## Choosing Hyperparameters


`winners-curse`


In [ ]:
rng = np.random.default_rng(774)
n_settings, n_test, true_error, trials = 50, 200, 0.25, 5000
scores = rng.binomial(n_test, true_error, size=(trials, n_settings)) / n_test
best = scores.min(axis=1)                               # the setting we would report
fresh = rng.binomial(n_test, true_error, size=trials) / n_test

print(f"true error of every setting:                  {true_error:.3f}")
print(f"reported error of the best of {n_settings} (mean):       {best.mean():.3f}")
print(f"same model, scored on a fresh test set (mean): {fresh.mean():.3f}")
print(f"chance the reported error is below 20%:       {np.mean(best < 0.20):.2f}")

## Cross-Validation in Practice


`fig-cross-validation`


In [ ]:
I, K_folds = 30, 5
caps = np.arange(1, 21)
rng = np.random.default_rng(774)

def cv_error(x, y, D, folds):
    """Average validation MSE over the K folds."""
    errs = []
    for val in folds:
        trn = np.setdiff1d(np.arange(len(x)), val)
        errs.append(np.mean((predict(fit(x[trn], y[trn], D), x[val], D) - y[val]) ** 2))
    return np.mean(errs)

x, y = sample(I, rng)
folds = np.array_split(rng.permutation(I), K_folds)
cv    = np.array([cv_error(x, y, D, folds) for D in caps])
train = np.array([np.mean((predict(fit(x, y, D), x, D) - y) ** 2) for D in caps])
true  = np.array([np.mean((predict(fit(x, y, D), x_grid, D) - mu(x_grid)) ** 2) + SIGMA**2 for D in caps])

fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(caps, train, color=ACCENT, label="training error")
ax.plot(caps, cv, color=STEEL, label="5-fold CV estimate")
ax.plot(caps, true, color="k", label="true expected test loss")
ax.plot(caps[cv.argmin()], cv.min(), "o", color=STEEL)
ax.plot(caps[true.argmin()], true.min(), "o", color="k")
ax.set(yscale="log", xlabel="Capacity $D$", ylabel="Mean squared error", xticks=caps[::2])
ax.legend()
fig.tight_layout()
plt.show()

# How reliable is the choice? Repeat over 200 fresh datasets.
picks, regret = [], []
for _ in range(200):
    x2, y2 = sample(I, rng)
    f2 = np.array_split(rng.permutation(I), K_folds)
    c = [cv_error(x2, y2, D, f2) for D in caps]
    t = [np.mean((predict(fit(x2, y2, D), x_grid, D) - mu(x_grid)) ** 2) for D in caps]
    picks.append(caps[np.argmin(c)]); regret.append(t[np.argmin(c)] - min(t))
picks = np.array(picks)
print(f"this dataset: CV chooses D = {caps[cv.argmin()]}, the truth prefers D = {caps[true.argmin()]}")
print(f"over 200 datasets: CV chooses D = 4 in {np.mean(picks == 4):.0%} of them and D = 3 to 7 in {np.mean((picks >= 3) & (picks <= 7)):.0%}")
print(f"median excess test error from CV's choice: {np.median(regret):.4f}  (noise alone is {SIGMA**2:.4f})")